# 04 · Text-Conditioned Retrain  (fix the modality gap + texture)

The Stage-2 diffusion model only learned CLIP *image*-embedding conditions, so it responds weakly to the *text* embeddings a crowd produces (RQ1 fidelity ~0.18, aggregators barely separate). This retrain **teaches the model to respond to text** by conditioning on a **mix** of each painting's CLIP image embedding **and** the CLIP *text* embedding of its WikiArt label (`--text-cond --p-text 0.5`), and simultaneously pushes quality (more data, bigger U-Net).

Saves to a **separate** path (`diffusion_v2`) so the baseline stays intact for a before/after RQ1 comparison.

> Runtime → **GPU (A100)** recommended (bigger U-Net + more data). ~15–20 min on A100, ~60 min on T4.

## 1. Clone & install

In [ ]:
!git clone --branch feature/crowd-driven-visual-generation https://github.com/nishant-kumar109/gen-ai-IISc.git
%cd gen-ai-IISc/projects/crowd-driven-visual-generation
!pip install -q datasets wandb open-clip-torch
import torch; print('cuda', torch.cuda.is_available(), '|',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Credentials & locate the VAE
W&B key + a **Write**-scope HF token (for §6 upload). Locates the frozen Stage-1 `vae.pt`.

In [ ]:
import os, getpass
from google.colab import drive; drive.mount('/content/drive')

wandb_key = getpass.getpass('W&B API key (Enter to skip): ').strip()
if wandb_key:
    import wandb; wandb.login(key=wandb_key); print('✓ W&B')
hf_token = getpass.getpass('HF token (Write scope): ').strip()
if hf_token:
    from huggingface_hub import login; login(token=hf_token); print('✓ HF')

VAE_CKPT = '/content/drive/MyDrive/crowdgen/vae/vae.pt'
if not os.path.exists(VAE_CKPT):
    from huggingface_hub import hf_hub_download, whoami
    VAE_CKPT = hf_hub_download(f"{whoami()['name']}/crowdgen-vae", 'vae.pt')
OUT = '/content/drive/MyDrive/crowdgen/diffusion_v2'; os.makedirs(OUT, exist_ok=True)
print('VAE:', VAE_CKPT, '\nOUT:', OUT)

## 3. Retrain — text-conditioned + bigger
`--text-cond` mixes image & label-text conditions (`--p-text 0.5`); `--label-col genre` uses WikiArt's content-oriented labels (try `style` too). Bigger run: `--limit 15000 --base 128 --epochs 120`. Watch `loss/eps_mse` and the `samples/live` grid (now generated from **text** conditions) in W&B.

In [ ]:
!python train_diffusion.py --vae {VAE_CKPT} --dataset huggan/wikiart --limit 15000 \
    --image-size 64 --batch 128 --epochs 120 --lr 2e-4 --base 128 \
    --text-cond --p-text 0.5 --label-col genre \
    --out {OUT} --guidance 3.0 --sample-steps 50 --wandb --sample-every 10

## 4. Inspect samples (text-conditioned)
Top = real paintings, bottom = generated from each painting's **label-text** embedding. Sharper/more on-label than the baseline means text conditioning is now working.

In [ ]:
from IPython.display import Image; Image(f'{OUT}/samples.png')

## 5. Upload to HF  *(separate repo — keeps the baseline)*

In [ ]:
import os
from huggingface_hub import HfApi, create_repo, whoami
repo_id = f"{whoami()['name']}/crowdgen-diffusion-v2"
create_repo(repo_id, repo_type='model', exist_ok=True, private=True)
api = HfApi()
api.upload_file(path_or_fileobj=f'{OUT}/diffusion.pt', path_in_repo='diffusion.pt',
                repo_id=repo_id, repo_type='model')
if os.path.exists(f'{OUT}/samples.png'):
    api.upload_file(path_or_fileobj=f'{OUT}/samples.png', path_in_repo='samples.png',
                    repo_id=repo_id, repo_type='model')
print('✓', f'https://huggingface.co/{repo_id}')

## 6. Re-run RQ1 on the retrained model — the before/after
Same evaluation as `03` §7, now on `diffusion_v2`. Compare these numbers to the baseline (fidelity ~0.18, aggregators barely separated). **If text conditioning worked, absolute fidelity should rise and the mean-vs-centroid gap should widen** — the decisive RQ1 result.

In [ ]:
!python evaluate.py --vae {VAE_CKPT} --diffusion {OUT}/diffusion.pt \
    --repeats 8 --n 300 --guidance 3.0 --steps 50 --out {OUT}/rq1
from IPython.display import Image, display
display(Image(f'{OUT}/rq1/rq1_fidelity.png')); display(Image(f'{OUT}/rq1/rq1_consistency.png'))

## Next
Compare v2's RQ1 numbers to the baseline (`FINDINGS.md`). If conditioning improved: re-run `03_crowd_to_image.ipynb` pointing at `diffusion_v2` for the qualitative themes/diversity grids, then add the **learnable aggregators** (DeepSets / attention) to complete the study. If it didn't move much, that's a legitimate finding too (scale-limited) — we document it and lean on the honest analysis.